# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema and is accessible via URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The Croissant Dataset metadata is available as attributes and as a json object
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")


## 2. Data Overview

Review available **record sets**, **fields**, and inspect their `@id` references, as well as fetch some example records.

In [ ]:
# List all record set @ids defined in the dataset
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
if len(record_sets) == 0:
    # If not directly provided, try to derive via the distribution/fileObject/record sets
    # This dataset exposes its main data as a single table
    from pprint import pprint
    print("No direct 'recordSet' entries found in top-level metadata. Inspecting available distributions:")
    pprint(metadata.to_json().get('distribution', []))
    # Printed for user inspection
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- {rs}")

The primary table for FAIR² datasets is usually named by convention and exposed as a main `RecordSet`. We'll examine the structure and load a few example records using their `@id`.

In [ ]:
# Find details of main table: The 'distribution' keys have fileObject references (pointing to the data)
main_dist = metadata.to_json().get('distribution', [])[0]
main_fileobject_id = main_dist.get('@id') if isinstance(main_dist, dict) else main_dist
print(f"Primary data FileObject @id: {main_fileobject_id}")

# Try loading the very first RecordSet
if len(record_sets):
    main_record_set_id = record_sets[0]
else:
    # Try to guess record set name from distribution id if not provided
    main_record_set_id = main_fileobject_id

print(f"Chosen record set @id: {main_record_set_id}\n")

# Print some records as dicts (the field keys will give us direct field @ids)
try:
    example_records = list(dataset.records(record_set=main_record_set_id))
    print(f"First record keys (field @ids): {list(example_records[0].keys())}")
    print(f"First record values: {list(example_records[0].values())}")
except Exception as e:
    print(f"Could not preview example records: {e}")

Below, we directly enumerate all the available fields (columns) in the main record set, along with their `@id`, `name`, and type descriptions where possible. This helps select fields for analysis.

In [ ]:
# Enumerate the available fields and their info from the dataset metadata
field_defs = []
all_fields = metadata.to_json().get('field', [])
if not all_fields:
    # Try to extract from 'recordSet' if not top-level
    possible_rs = metadata.to_json().get('recordSet', [])
    if possible_rs and isinstance(possible_rs, list):
        for rs in possible_rs:
            if 'field' in rs:
                all_fields.extend(rs['field'])
# Typical FAIR2 packages list fields at top level, let's enumerate
print(f"Number of fields/columns: {len(all_fields)}\n")
for field in all_fields:
    field_id = field.get('@id', '<no_id>')
    field_name = field.get('name', None)
    dtype = field.get('dataType', None)
    print(f"@id: {field_id}, name: {field_name}, dataType: {dtype}")

## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. All data references will use their canonical `@id`s from above.

In [ ]:
# We focus on the single main record set identified above
record_sets = [main_record_set_id]
dataframes = {}

for record_set in record_sets:
    print(f"Loading records from record set {record_set} ...")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Loaded {len(df)} records. Columns (field @ids):\n{df.columns.tolist()}\n")

# Show the first few rows
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We will select a numeric field (e.g., protein MW or normalized abundance) to explore distribution, filter records, and normalize data. All operations will use field `@id`.

In [ ]:
# Choose a numeric field @id -- for demonstration, try to use a likely main numeric column
# Update this variable with the correct field from your earlier field listing
possible_numeric_fields = [c for c in dataframes[main_record_set_id].columns if ('mw' in c.lower() or 'weight' in c.lower() or 'abundance' in c.lower() or 'count' in c.lower())]
if len(possible_numeric_fields) == 0:
    print("No obvious numeric fields found. Please update `numeric_field` manually using the column list above.")
    numeric_field = dataframes[main_record_set_id].columns[0]  # Pick first
else:
    numeric_field = possible_numeric_fields[0]
print(f"Field selected for EDA: {numeric_field}")

# Sanity: attempt to coerce the field to numeric
df = dataframes[main_record_set_id]
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)  # Top quartile value as threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records\n")
print(filtered_df.head())

# Normalize data
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records (first 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g., 'description', 'accession', etc.)
possible_group_fields = [c for c in df.columns if ('desc' in c.lower() or 'accession' in c.lower() or 'sample' in c.lower() or 'type' in c.lower())]
group_field = possible_group_fields[0] if possible_group_fields else None

if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing mean values):")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization

Below, visualize data distributions and simple relationships between fields, using the chosen numeric field. (If you know the biology, try comparing abundance vs. MW, for example.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If available, plot relationship between two numeric fields
other_numeric_fields = [c for c in df.columns if c != numeric_field and df[c].dtype.kind in 'fi']
if len(other_numeric_fields):
    y_numeric = other_numeric_fields[0]
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=df[numeric_field], y=df[y_numeric])
    plt.title(f"{numeric_field} vs. {y_numeric}")
    plt.xlabel(numeric_field)
    plt.ylabel(y_numeric)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we:

- Loaded and explored the FAIR² dataset describing mass spectrometry results from human mast cell derived extracellular vesicles, using its Croissant schema.
- Identified and accessed data via entity `@id` values, listing available fields and their identifiers.
- Performed exploratory analysis on a key quantitative field (e.g., protein abundance/MW), including filtering, normalization, and grouping by categorical variables.
- Visualized distributions and possible relationships between numeric attributes.

Further biological or machine learning analyses can now build on this cleaned and well-understood data foundation.